# Evaluación de modelos fine-tuneados con adapters

In [1]:
import os

# Force PyTorch to allow full object loading (Must be before torch imports)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

In [2]:
import torch
from pathlib import Path
from nemo.collections.asr.parts.utils.manifest_utils import read_manifest

OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


In [3]:
import nemo.collections.asr as nemo_asr

MODEL = "nvidia/canary-1b-v2"
model = nemo_asr.models.ASRModel.from_pretrained(MODEL)

[NeMo I 2026-04-10 03:41:21 mixins:184] Tokenizer CanaryBPETokenizer initialized with 16384 tokens


[NeMo W 2026-04-10 03:41:21 modelPT:176] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 4
    pin_memory: true
    prompt_format: canary2
    max_duration: 40.0
    min_duration: 0.01
    text_field: answer
    lang_field: target_lang
    use_bucketing: true
    max_tps: null
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: null
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2026-04-10 03:41:21 modelPT:183] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation

[NeMo I 2026-04-10 03:41:24 save_restore_connector:146] Restoration will occur within pre-extracted directory : `/tmp/tmpla_vulz_`.
[NeMo I 2026-04-10 03:41:25 mixins:184] Tokenizer SentencePieceTokenizer initialized with 16384 tokens


[NeMo W 2026-04-10 03:41:26 modelPT:176] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 40.0
    min_duration: 0.1
    text_field: answer
    batch_duration: null
    max_tps: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: null
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2026-04-10 03:41:26 modelPT:183] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validatio

[NeMo I 2026-04-10 03:41:29 save_restore_connector:286] Model EncDecCTCModelBPE was successfully restored from /tmp/tmpla_vulz_.
[NeMo I 2026-04-10 03:41:30 save_restore_connector:286] Model EncDecMultiTaskModel was successfully restored from /home/umoqnier/.cache/huggingface/hub/models--nvidia--canary-1b-v2/snapshots/87bc52657add533cd0156b3fc1aef027280754bf/canary-1b-v2.nemo.


In [4]:
import re
from nemo.collections.asr.metrics.wer import word_error_rate
from torchmetrics.text import SacreBLEUScore

def preprocess_text(text):
    text = re.sub(r"[^\w\s]", "", text)
    text = text.lower()
    return text

def get_eval_data(manifest_path: str, max_samples: int = None):
    data = read_manifest(manifest_path)
    if max_samples is not None:
        file_paths = [x["audio_filepath"] for x in data[:max_samples]]
        gt_texts = [x["text"] for x in data[:max_samples]]
    else:
        file_paths = [x["audio_filepath"] for x in data]
        gt_texts = [x["text"] for x in data]
    return file_paths, gt_texts


def run_predictions(model, file_paths) -> list:
    if torch.cuda.is_available():
        model = model.cuda()
        model = model.to(torch.bfloat16)
    
    preds = model.transcribe(
        file_paths,
        pnc="no",
        task="ast",
        source_lang="en",
        target_lang="es",
        batch_size=32,
    )
    return [preprocess_text(p.text) for p in preds]



def eval_model(model, predictions, gt_texts) -> dict:
    wer = word_error_rate(predictions, gt_texts)
    sacrebleu = SacreBLEUScore(n_gram=4)
    # bleu = sum(scores) / len(scores)
    sacrebleu.update(predictions, gt_texts)
    bleu = sacrebleu.compute()
    return {"wer": wer, "bleu": bleu}

### Evaluando Nemo sin finetuning

In [5]:
test_manifest = "combined_data/train_map_manifest_skiped_lines.json"

file_paths, gt_texts = get_eval_data(test_manifest, max_samples=10)

In [6]:
preds = run_predictions(model, file_paths)

[NeMo W 2026-04-10 03:41:31 aed_multitask_models:607] Chunking is disabled. Please pass a single audio file or set batch_size to 1
[NeMo W 2026-04-10 03:41:31 dataloader:879] The following configuration keys are ignored by Lhotse dataloader: trim_silence,enable_chunking
[NeMo W 2026-04-10 03:41:31 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:06,  6.82s/it]


In [7]:
preds[:3]

['mi nombre es agustín curikeo kintriqueo pinita y ngai mi nombre es kangura pinita mi nombre es mleta inchiao pinita y ngai mi nombre es longo y josé sangre moltfinao pinita y tache',
 'sí si no puedo levantarme puedo levantarme puedo levantarme puedo levantarme puedo levantarme puedo levantarme puedo levantarme puedo levantarme puedo levantarme puedo levantarme puedo levantarme',
 'en chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba chamba c

In [9]:
result = eval_model(
    model, preds, gt_texts
)

In [10]:
print(result)

{'wer': 2.4036511156186613, 'bleu': tensor(0.)}


## Cargando modelo fine-tuneado

In [23]:
!ls models

'canary_adapter_epoch=09.ckpt'	 last.ckpt
'canary_adapter_epoch=19.ckpt'	 last_pnc.ckpt
 canary_map_bs8_steps50.pt	 mapuche_bs8_epochs80.pt


### Probando un modelo horrendo

In [11]:
# This converts the standard architecture to an adapter-compatible one
model.replace_adapter_compatible_modules()

[NeMo I 2026-04-10 03:43:05 adapter_mixins:169] Swapping class nemo.collections.asr.modules.conformer_encoder.ConformerEncoder with adapter compatible class: nemo.collections.asr.modules.conformer_encoder.ConformerEncoderAdapter
[NeMo I 2026-04-10 03:43:05 adapter_mixins:169] Swapping class nemo.collections.asr.modules.transformer.transformer.TransformerDecoderNM with adapter compatible class: nemo.collections.asr.modules.transformer.transformer.TransformerDecoderNMAdapter
[NeMo I 2026-04-10 03:43:05 adapter_mixins:169] Swapping class nemo.collections.asr.modules.transformer.transformer_decoders.TransformerDecoder with adapter compatible class: nemo.collections.asr.modules.transformer.transformer_decoders.TransformerDecoderAdapter


In [12]:
MODELS_PATH = "models"

def load_model(model_name):
    model_path = Path(MODELS_PATH) / model_name
    print(f"Load Map adapters from {model_path}")
    model.load_adapters(model_path)
    # Put the model in inference mode
    model.eval()
    return model

In [13]:
model = load_model("mapuche_bs8_epochs80.pt")

Load Map adapters from models/mapuche_bs8_epochs80.pt


In [14]:
preds = run_predictions(model, file_paths)

[NeMo W 2026-04-10 03:43:10 aed_multitask_models:607] Chunking is disabled. Please pass a single audio file or set batch_size to 1


[NeMo W 2026-04-10 03:43:10 dataloader:879] The following configuration keys are ignored by Lhotse dataloader: trim_silence,enable_chunking
[NeMo W 2026-04-10 03:43:10 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 0it [00:00, ?it/s][NeMo W 2026-04-10 03:43:10 attention_adapter_mixin:125] No adapter compatible with the current module. Skipping adapter forward pass.
Transcribing: 1it [00:06,  6.67s/it]


In [15]:
result = eval_model(
    model,  preds, gt_texts
)

In [16]:
result

{'wer': 4.9208924949290065, 'bleu': tensor(0.)}

In [17]:
for pred, gt in zip(preds, gt_texts):
    print(f"GT: {gt}")
    print(f"Pred: {pred}")
    print("---")

GT: hola hermana mi nombre agustín curiqueo quintriqueo se dice mi nombre mi tierra se llama cancura allí están mi papá mis mamás mi hermano mi hermana el principal de mi familia se llama josé sangre mollfunao mi viejo es ese pues así es pues hermana
Pred: el señor de la mujer el señor de la mujer el señor de la mujer el señor de la mujer el señor de la mujer el señor de la mujer el señor de la mujer el señor de la mujer el señor de la mujer el señor de la mujer el señor de la mujer el señor de la mujer el señor de la mujer el señor de la mujer
---
GT: ya yo también mi nombre me llamo yo soy llamada maría isabel coñoepán me llamo yo mi procedencia es huapi de huapi provengo yo de mi de mi comunidad mi sector se llama cahuemu cahuemu se llama ahora estoy acá en temuco estoy aqui ahora pero mi tierra está allá en cahuemu isla yo soy isleña
Pred: sí así es como se hace así es como se hace así es como se hace así es como se hace así es como se hace así es como se hace así es como se hace a